# Lab 1.2 &mdash; The Four Building Blocks

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Write tools with a real argument schema, and docstrings the model actually reads
- Bound memory with `trim_messages` &mdash; including the token counter that breaks on this model
- Turn a goal into a dependency-ordered plan with `with_structured_output` and Pydantic
- Assemble all four blocks into one agent over the case file

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **Builds on Lab 1.1.** The loop you wrote there is the fourth block; here you build
> the other three as first-class LangChain objects and wire them together.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.1
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool

SYSTEM = ("You are a payments operations analyst. Answer only from the data you are given. "
          "If you do not have the data, say so.")

def run_tool_calls(ai_message, tools: dict) -> list:
    out = []
    for call in ai_message.tool_calls:
        try:
            result = tools[call["name"]].invoke(call["args"])
        except Exception as exc:
            result = f"tool error: {type(exc).__name__}: {exc}"
        out.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return out

print("Lab 1.1 helpers loaded")

## Concept

Each block patches one thing a model cannot do on its own &mdash; and each has a LangChain object
that *is* that block:

| Block | The gap it closes | The object |
|---|---|---|
| **LLM** | judgement under ambiguity | `ChatOpenAI` |
| **Memory** | the call is stateless | a message list + `trim_messages` |
| **Tools** | the model cannot read or change anything | `@tool` / `StructuredTool` |
| **Planning** | a goal is not a sequence of steps | `with_structured_output(...)` |

Miss one and you have a pipeline with a model in it &mdash; often the right build, but not an agent.

## Section 1 &mdash; Tools: the docstring is the instruction

`@tool` turns a function into something the model can be offered. Three parts of it are read by
the model and by nothing else:

- the **name** &mdash; taken from the function name,
- the **description** &mdash; taken from the docstring,
- the **argument schema** &mdash; inferred from your type hints, or given explicitly with Pydantic.

Get the docstring wrong and the model picks the wrong tool. Lab 1.3 measures exactly that. Here,
write one that says both what the tool is for *and* what it is not for.

In [ ]:
from pydantic import BaseModel, Field

class ReleaseArgs(BaseModel):
    """Arguments for release_payment."""
    ref: str = Field(description="The payment reference, e.g. 'PMT-1003'")
    approved_by: str = Field(description="Name of the human who approved the release")


@tool(args_schema=ReleaseArgs)
def release_payment(ref: str, approved_by: str) -> str:
    """Release one held payment for settlement, on a named human's authority.

    Use only after a human has approved the release. Never use it to clear a sanctions
    hold, and never invent an approver.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    if record["reason_code"] in NEEDS_HUMAN and not approved_by:
        return f"refused: {record['reason_code']} needs a named human approver"
    return f"released {ref} on the authority of {approved_by}"


@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {t.name: t for t in (lookup_payment, policy_for, release_payment)}

In [ ]:
# --- Self-check: Section 1   (inspects the tool objects -- no model call)
def _doc(name):
    d = (TOOLS[name].description or "").strip()
    if d == "BLANK":
        raise NameError("release_payment still has the placeholder docstring")
    return d

check("@tool takes its name from the function",
      lambda: TOOLS["lookup_payment"].name == "lookup_payment")
check("the schema was inferred from the type hints",
      lambda: "ref" in TOOLS["lookup_payment"].args)
check("release_payment declares both arguments",
      lambda: set(TOOLS["release_payment"].args) == {"ref", "approved_by"})
check("the Field descriptions reached the schema",
      lambda: "approver" in json.dumps(TOOLS["release_payment"].args).lower()
              or "approved" in json.dumps(TOOLS["release_payment"].args).lower())
check("release_payment has a real description",
      lambda: len(_doc("release_payment")) > 40,
      "the docstring is the only thing the model reads when choosing this tool")
check("the description says when NOT to use it",
      lambda: any(w in _doc("release_payment").lower() for w in ("never", "not ", "only")),
      "a tool that only says what it does gets called when it should not be")
check("a tool invoked with a dict returns its result",
      lambda: "INSUFFICIENT_FUNDS" in TOOLS["lookup_payment"].invoke({"ref": "PMT-1002"}))
check("an unknown reference does NOT raise",
      lambda: "no payment found" in TOOLS["lookup_payment"].invoke({"ref": "PMT-9999"}),
      "a raising tool aborts the whole agent run")

## Section 2 &mdash; Memory: `trim_messages`, and the counter that breaks

Unbounded history is the classic failure: fine in the demo, degraded and expensive by week two.
LangChain's `trim_messages` bounds it for you &mdash; but it needs to know how to count tokens, and
**the obvious answer does not work here**:

```python
trim_messages(msgs, max_tokens=200, token_counter=llm)   # NotImplementedError on this model
```

`token_counter=llm` asks the model class to count, and `langchain-openai` only knows how to do
that for models `tiktoken` has an encoding for. `qwen36-35b-a3b-lab` is not one. The fix is
`count_tokens_approximately`, which is a plain function over the message text.

This is not a quirk of our sandbox &mdash; it is what happens to every self-hosted or gateway-served
model, and it is the kind of thing that only shows up under load.

In [ ]:
from langchain_core.messages.utils import count_tokens_approximately

def bounded(messages: list, max_tokens: int = 120) -> list:
    """Keep the system message and as many recent turns as fit inside `max_tokens`."""
    from langchain_core.messages import trim_messages
    return trim_messages(
        messages,
        max_tokens=max_tokens,
        token_counter=count_tokens_approximately,   # a plain function over the text
        strategy="last",              # keep the END of the conversation, not the start
        include_system=True,          # never drop the instructions
        start_on="human",             # a valid history starts on a human turn
        allow_partial=False,
    )

In [ ]:
# --- Self-check: Section 2   (trim_messages is pure -- no model call)
def _long_history():
    msgs = [SystemMessage(SYSTEM), HumanMessage("Investigate PMT-1003, it is held.")]
    for i in range(12):
        msgs.append(AIMessage(f"step {i}: " + "checking the ledger. " * 12))
        msgs.append(HumanMessage(f"and then? ({i})"))
    return msgs

check("the history is bounded",
      lambda: count_tokens_approximately(bounded(_long_history())) <= 130,
      "trim_messages needs a token_counter it can actually call on this model")
check("trimming actually dropped turns",
      lambda: len(bounded(_long_history())) < len(_long_history()))
check("the system message survives",
      lambda: bounded(_long_history())[0].type == "system",
      "include_system=True -- dropping the instructions is the worst possible trim")
check("what survives is the END of the conversation",
      lambda: bounded(_long_history())[-1].content == _long_history()[-1].content,
      'strategy="last" keeps recent turns; "first" would keep the stale ones')
check("a short conversation is left untouched",
      lambda: len(bounded([SystemMessage(SYSTEM), HumanMessage("hi")])) == 2)

## Section 3 &mdash; Planning: a goal is not a sequence

Decomposition is only half of it. The steps have **dependencies**, and running them out of order
is one of the quieter ways an agent wastes a budget.

`with_structured_output(Plan)` makes the model return a **`Plan` object**, not prose that you then
have to parse. You declare the shape; LangChain gives the model the schema and validates what
comes back. Define the schema first, then write the ordering.

In [ ]:
from typing import List

class Step(BaseModel):
    """One step of an investigation plan."""
    name: str = Field(description="Short snake_case name for this step")
    depends_on: List[str] = Field(default_factory=list,
                                  description="Names of steps that must finish before this one")
    tool: str = Field(description="Which tool this step calls, or 'none'")


class Plan(BaseModel):
    """An ordered investigation plan for one payment exception."""
    goal: str = Field(description="The question the plan answers, in one line")
    steps: List[Step] = Field(description="The steps, which may be given in any order")


def order_steps(plan: Plan) -> list[str]:
    """Return a runnable order for plan.steps, respecting depends_on.

    Raises ValueError if the dependencies cannot be satisfied (a cycle, or a missing step).
    """
    deps = {s.name: list(s.depends_on) for s in plan.steps}
    ordered: list[str] = []
    done: set[str] = set()
    while len(ordered) < len(deps):
        progressed = False
        for name, d in deps.items():
            if name in done:
                continue
            if all(x in done for x in d):        # every dependency already ordered
                ordered.append(name)
                done.add(name)
                progressed = True
        if not progressed:
            raise ValueError("cycle or missing dependency in plan")
    return ordered

In [ ]:
# --- Self-check: Section 3   (Pydantic objects only -- no model call)
HAND_PLAN = Plan(goal="Decide what to do about PMT-1003", steps=[
    Step(name="decide_action", depends_on=["read_payment", "read_policy"], tool="none"),
    Step(name="read_policy",   depends_on=["read_payment"], tool="policy_for"),
    Step(name="read_payment",  depends_on=[], tool="lookup_payment"),
    Step(name="write_note",    depends_on=["decide_action"], tool="none"),
])

def _cycles():
    try:
        order_steps(Plan(goal="g", steps=[Step(name="a", depends_on=["b"], tool="none"),
                                          Step(name="b", depends_on=["a"], tool="none")]))
        return False
    except ValueError:
        return True

check("the schema declares a goal and steps",
      lambda: set(Plan.model_fields) == {"goal", "steps"})
def _desc(field: str) -> str:
    d = (Step.model_fields[field].description or "").strip()
    if d == "BLANK":
        raise NameError(f"{field} still has the placeholder description")
    return d

check("every step field carries a description the model can read",
      lambda: all(_desc(f) for f in Step.model_fields),
      "with_structured_output sends these descriptions to the model as the schema")
check("depends_on says what the names REFER to",
      lambda: "step" in _desc("depends_on").lower(),
      "the model has to know these are other steps' names, not tool names")
check("dependencies come before dependants",
      lambda: order_steps(HAND_PLAN).index("read_payment") < order_steps(HAND_PLAN).index("read_policy"))
check("every step is scheduled exactly once",
      lambda: sorted(order_steps(HAND_PLAN)) == sorted(s.name for s in HAND_PLAN.steps))
check("a plan given out of order is still ordered correctly",
      lambda: order_steps(HAND_PLAN)[0] == "read_payment")
check("an impossible plan raises rather than half-running", _cycles)

## Section 4 &mdash; Assemble the four blocks

One object now holds all four: the model, the bound tools, the bounded history, and a plan.
`answer()` runs the loop until the model replies without asking for a tool.

In [ ]:
class MiniAgent:
    """LLM + Memory + Tools + Planning, assembled by hand. `create_agent` is this, hardened."""

    def __init__(self, tools: dict, max_tokens: int = 600, max_steps: int = 6):
        self.tools = tools
        self.max_tokens, self.max_steps = max_tokens, max_steps
        self.history = [SystemMessage(SYSTEM)]

    def tool_list(self) -> list:
        """The tool OBJECTS this agent may call -- bind_tools() needs the objects, not names."""
        return list(self.tools.values())

    @property
    def model(self):
        """The model, bound to this agent's tools so it can answer with a tool call."""
        return get_llm().bind_tools(self.tool_list())

    def plan(self, goal: str) -> Plan:
        """Ask the model for a Plan object -- not prose about a plan."""
        return get_llm().with_structured_output(Plan).invoke(
            "Produce an investigation plan for this goal. Steps must name their dependencies. "
            f"Available tools: {list(self.tools)}.\n\nGOAL: {goal}")

    def answer(self, question: str) -> str:
        self.history.append(HumanMessage(question))
        for _ in range(self.max_steps):
            self.history = bounded(self.history, self.max_tokens)
            ai = self.model.invoke(self.history)
            self.history.append(ai)
            if not ai.tool_calls:
                return ai.content
            self.history.extend(run_tool_calls(ai, self.tools))
        return "(step budget spent)"

In [ ]:
# --- Self-check: Section 4   (structure only -- .model builds an object, it does not call out)
check("MiniAgent starts with just the system message",
      lambda: [m.type for m in MiniAgent(TOOLS).history] == ["system"])
check("all three tools were handed to the agent",
      lambda: set(MiniAgent(TOOLS).tools) == {"lookup_payment", "policy_for", "release_payment"})
check("the agent hands bind_tools the tool objects, not their names",
      lambda: all(hasattr(t, "invoke") and hasattr(t, "name") for t in MiniAgent(TOOLS).tool_list()),
      "bind_tools() needs the @tool objects -- a list of strings binds nothing")
check("all three tools are offered to the model",
      lambda: {t.name for t in MiniAgent(TOOLS).tool_list()}
              == {"lookup_payment", "policy_for", "release_payment"})

## Run it for real

First a plan as a typed object, then the assembled agent answering a real question.

In [ ]:
if llm_ready():
    def _plan():
        agent = MiniAgent(TOOLS)
        p = agent.plan("Decide what to do about PMT-1003, which is held.")
        print("goal:", p.goal)
        for s in p.steps:
            print(f"  {s.name:16} tool={s.tool:16} after={s.depends_on}")
        print("\nrunnable order:", order_steps(p))
        return agent
    agent = guard(_plan)

In [ ]:
if llm_ready() and agent is not None:
    def _answer():
        print(agent.answer("Why is PMT-1003 held, and what must we do about it?")[:400])
        print("\n--- history after the run ---")
        show_messages(agent.history)
    guard(_answer)

### Read it

The plan came back as a `Plan`, so `order_steps` could run over it directly &mdash; no parsing, no
"the model used a different heading this time". That is the whole argument for structured output,
and Module 2 shows what the alternative costs.

Watch the history: it is bounded, so a long investigation cannot grow the bill without limit. And
notice which tool the model did *not* reach for. `release_payment` says "never use it to clear a
sanctions hold" in its docstring, and that sentence is the only control that stopped it.

In [ ]:
score()

## Your turn

1. Delete the second paragraph of `release_payment`'s docstring, re-run, and ask the agent to
   release PMT-1005. Put the sentence back once you have seen what happens.
2. `bounded()` uses `strategy="last"`. Switch it to `"first"` and ask a follow-up question.
   Explain, in one line, why keeping the *start* of a conversation is almost always wrong for an
   agent and almost always right for a chat product.
3. `MiniAgent.plan` never uses the plan &mdash; `answer()` just loops. Make `answer()` follow the
   ordered plan instead, and note the first thing that breaks.